In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
config.repeats = 5
config.gpu_id = 1

# config = CNNExperiments()
exp = ExperimentRun(config=config)

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
models = [
    "EleutherAI/gpt-neo-125M",
    "facebook/opt-125m",
    "facebook/opt-350m",
    "cerebras/Cerebras-GPT-111M",
    "microsoft/deberta-base",
    "T5-small",
    "t5-base",
    "distilbert/distilgpt2",
    "openai-community/gpt2",
]
model = models[-5]
batch = 15
optimizer = "AdamW"
gpu_id = 1
task_id = None
in_docker = True

In [5]:
# for model in models:
#     for batch in range(10, 20, 5):
#         for i in range(config.repeats):
#             exp.add_task(
#                 model_name=model,
#                 batch_size=batch,
#                 optimizer=optimizer,
#                 gpu_id=gpu_id,
#                 task_id=task_id,
#             )
# for batch in range(5, 20, 5):
#     for i in range(1):
#         exp.add_task(
#             model_name=model,
#             batch_size=batch,
#             optimizer=optimizer,
#             gpu_id=gpu_id,
#             task_id=task_id,
#         )
exp.add_task(
    model_name=model,
    batch_size=batch,
    optimizer=optimizer,
    gpu_id=gpu_id,
    task_id=task_id,
)



## Measure Ground Truth and Estimated Memory for Each job

In [6]:
exp.run_group_truth(in_docker=in_docker)

100%|██████████| 1/1 [00:00<00:00, 64.90it/s]


=============== Start massively run for GPU train ======================


100%|██████████| 1/1 [03:10<00:00, 190.71s/it]
0it [00:00, ?it/s]


## Estimate Max GPU Memory by DNNmem

In [7]:
exp.run_estimation(estimators=[SummarySectionName.DNNmem], in_docker=in_docker)

================== Create docker containers ==================


100%|██████████| 75/75 [02:31<00:00,  2.02s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 75/75 [34:47<00:00, 27.83s/it]
0it [00:00, ?it/s]

================== Statistics ==================
Run(success/total): 75/75


## Estimate Max GPU Memory by SchedTune

In [5]:
exp.run_estimation(estimators=[SummarySectionName.schedtune], in_docker=True)

================== Create docker containers ==================


100%|██████████| 75/75 [02:31<00:00,  2.02s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


0it [00:00, ?it/s]
100%|██████████| 75/75 [34:02<00:00, 27.23s/it]

================== Statistics ==================
Run(success/total): 75/75


In [21]:
exp.statistics()
exp.to_evaluation_result()

100%|██████████| 2/2 [00:00<00:00, 6528.10it/s]


=============== Statistics for Transformer-Exp ==================
train: 2/2
config: 2/2
groundtruth: 2/2
solution: 2/2
schedtune: 0/2
DNNmem: 0/2
LLmem: 0/2


100%|██████████| 2/2 [00:00<00:00, 1552.58it/s]


[{'tool': 'solution',
  'memory': 8820621312,
  'oom': False,
  'runtime': 55455698160,
  'ground': 9498001408,
  'error': 0.07131817178185029,
  'real_oom': False,
  'correct_estimation': True,
  '2nd verification': {'oom': True, 'error': None}},
 {'tool': 'solution',
  'memory': 5207228416,
  'oom': False,
  'runtime': 49682100301,
  'ground': 9244246016,
  'error': 0.4367059891107078,
  'real_oom': False,
  'correct_estimation': True,
  '2nd verification': {'oom': True, 'error': None}}]